# Task 3 — PF grant recipients: classify, verify EINs, exact-match blanks

**Milestone 2, Pharos Intelligence (DS539).**

Input is the 990-PF grants-paid table (`pf_p14_t03_supplementaryinformation`, ~2.2M grant rows).
Each row is one grant from a foundation (the *filer*, whose EIN is always known) to a *recipient*,
whose EIN is the gap this notebook works on.

| Step | What it does |
|---|---|
| 1 | Download and clean the IRS EO Business Master File (BMF), Publication 78 and the auto-revocation list |
| 2 | Build one reference index of known organizations: BMF + Pub 78 + our own 990 and 990-PF filers |
| 3 | Classify every recipient: organization, government, individual, foreign, unspecified ("SEE ATTACHED"), unidentifiable |
| 4 | Clean the recipient EINs that were reported and check them against the reference (does the EIN exist, does the name agree) |
| 5 | For blank or bad EINs: exact match on normalized name + ZIP5, then name + city + state. Only a single candidate counts as a match |
| 6 | Match-rate table (rows and dollars), MA slice, outputs for task 4 (fuzzy matching) and a spot-check sample |

Individuals and foreign recipients are set aside **before** computing the rate, as in the team data dictionary (Joins sheet).

Run top to bottom. Edit only the **Config** cell.

## Config

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Folder that holds the Pharos CSV extracts, laid out as on the team drive:
#   DATA_DIR/irs990_2024_2025/irs990_f9_p00_t00_header_2024_2025.csv
#   DATA_DIR/irs990pf_2024_2025/irs990pf_pf_p14_t03_supplementaryinformation_2024_2025.csv  ...
# If a file is not at that path, the Setup cell searches DATA_DIR (and then all of MyDrive) for it by name.
DATA_DIR = Path("/content/drive/MyDrive/DS539") if IN_COLAB else Path.home() / "Desktop" / "DS539"

FILES = {
    "pf_grants":   "irs990pf_2024_2025/irs990pf_pf_p14_t03_supplementaryinformation_2024_2025.csv",
    "pf_header":   "irs990pf_2024_2025/irs990pf_pf_p00_t00_header_2024_2025.csv",
    "f990_header": "irs990_2024_2025/irs990_f9_p00_t00_header_2024_2025.csv",
}

# Scope. Team decision: match on the FULL national dataset. EIN matching against the BMF / Pub 78 is national and
# many grants cross state lines, so filtering first would drop real matches. MA is a flag (`recipient_in_ma`), and
# the match-rate table reports overall and MA side by side. The MA filter happens later, at the ACS join.
FILER_STATES = None       # leave None (kept only for quick debugging runs)
RECIPIENT_STATES = None   # leave None (kept only for quick debugging runs)

NAME_AGREE_THRESHOLD = 60  # token_set_ratio below this flags "EIN exists but name disagrees"
FORCE_DOWNLOAD = False     # True re-downloads BMF / Pub 78 even if cached
SPOT_CHECK_PER_METHOD = 20

## Setup

In Colab this mounts Google Drive and installs `rapidfuzz` / `pyarrow`. Then it finds each input file:
first at `DATA_DIR / FILES[...]`, then anywhere under `DATA_DIR`, then anywhere under `MyDrive`
(files someone shared with you only show up there after **Add shortcut to Drive**).
Outputs go to `task3_outputs/` next to the data folders.

In [ ]:
import subprocess

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "rapidfuzz", "pyarrow"], check=True)

import csv, io, json, re, time, unicodedata, zipfile
from datetime import date

import numpy as np
import pandas as pd
import requests
from rapidfuzz import fuzz

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


def locate(rel: str) -> Path:
    p = DATA_DIR / rel
    if p.exists():
        return p
    name = Path(rel).name
    roots = [DATA_DIR] + ([Path("/content/drive/MyDrive")] if IN_COLAB else [])
    for root in roots:
        if root.exists():
            print(f"  searching {root} for {name} ...")
            hit = next(root.rglob(name), None)
            if hit:
                return hit
    raise FileNotFoundError(
        f"Could not find {name}. Upload the Pharos CSV folders to Drive and set DATA_DIR in the Config cell "
        f"(currently {DATA_DIR}).")


PATHS = {k: locate(v) for k, v in FILES.items()}
for k, p in PATHS.items():
    print(f"{k:12s} {p}  ({p.stat().st_size / 1e6:,.0f} MB)")

BASE_DIR = PATHS["pf_grants"].parent.parent  # folder that contains irs990pf_2024_2025/
OUT_DIR = BASE_DIR / "task3_outputs"
REF_DIR = OUT_DIR / "reference"
REF_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs go to", OUT_DIR)

## Normalization helpers

The same functions are applied to the grant rows and to every reference source, so keys line up.

* **Name**: strip accents, upper case, drop `C/O ...` and `ATTN ...` tails, `&` → `AND`, drop apostrophes
  (`WOMEN'S` → `WOMENS`), other punctuation → space, drop leading `THE` and trailing `, THE`,
  drop trailing legal suffixes (`INC`, `CORP`, `LLC`, `LTD`, ...). Words like `FOUNDATION` are kept: they distinguish orgs.
* **City**: same cleanup, plus leading `ST`/`MT`/`FT` → `SAINT`/`MOUNT`/`FORT`.
* **ZIP5**: digits only; a 4- or 8-digit value is treated as a ZIP that lost its leading zero (common for New England).
* **EIN**: see the EIN cell below.

In [ ]:
US_STATE_CODES = set("""
AL AK AZ AR CA CO CT DE DC FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND
OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY PR VI GU AS MP FM MH PW AA AE AP
""".split())

LEGAL_SUFFIX_RE = r"(?:\s+(?:INC|INCORPORATED|CORP|CORPORATION|CO|COMPANY|LLC|L L C|LTD|LIMITED|PLLC|PC|NFP))+$"


def strip_accents(x: str) -> str:
    if not x or x.isascii():
        return x
    return unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("ascii")


def _blank(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.strip()


def norm_name(s: pd.Series) -> pd.Series:
    s = _blank(s).map(strip_accents).str.upper()
    s = s.str.replace(r"\b(?:C/O|C\\O|ATTN:?|ATTENTION:?)(?:\s.*)?$", "", regex=True)
    s = s.str.replace("&", " AND ", regex=False)
    s = s.str.replace(r"['’`]", "", regex=True)
    s = s.str.replace(r"[^A-Z0-9 ]", " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    s = s.str.replace(r"^THE\s+", "", regex=True)
    s = s.str.replace(r"\s+THE$", "", regex=True)
    s = s.str.replace(LEGAL_SUFFIX_RE, "", regex=True)
    return s.str.strip()


def _compact(s: pd.Series) -> pd.Series:
    return s.str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)


def combine_name_lines(l1: pd.Series, l2: pd.Series) -> pd.Series:
    """Best single name from the two IRS name lines (raw text in, raw text out).

    * line 2 repeats the whole name (line 1 was cut at 35 chars): use line 2
    * line 2 is a cut-off copy of line 1 ("CORNELL LAB OF ORNITHOLOGY" / "CORNELL LAB OF ORNIT"): use line 1
    * line 2 starts lower-case right after a letter (a word split by the 35-char limit,
      e.g. "...Associatio" + "n Inc"): join without a space
    * otherwise: join with a space
    """
    a, b = _blank(l1), _blank(l2)
    ca, cb = _compact(a), _compact(b)
    repeat = pd.Series([bool(x) and y.startswith(x) for x, y in zip(ca, cb)], index=a.index)
    l2_is_prefix = pd.Series([bool(y) and x.startswith(y) for x, y in zip(ca, cb)], index=a.index)
    split_word = a.str.contains(r"[A-Za-z]$") & b.str.contains(r"^[a-z]")
    out = np.where(b == "", a, np.where(repeat, b, np.where(l2_is_prefix, a,
                   np.where(split_word, a + b, a + " " + b))))
    return pd.Series(out, index=a.index).str.strip()


DBA_RE = r"\b(?:D\s?/?\s?B\s?/?\s?A|A\s?/?\s?K\s?/?\s?A|F\s?/?\s?K\s?/?\s?A|DOING BUSINESS AS|FORMERLY)\b"


def name_variants(key: pd.Series, l1: pd.Series, l2: pd.Series) -> pd.DataFrame:
    """Long table (key, name_norm) with every reasonable reading of a two-line name:
    line 1, line 2, the combined name, line1+' '+line2, line1+line2 (no space, when line 1 is
    long enough to have been cut), and each side of a D/B/A, A/K/A or F/K/A split."""
    a, b = _blank(l1), _blank(l2)
    comb = combine_name_lines(a, b)
    long1 = a.str.len() >= 30
    parts = [a, b, comb, (a + " " + b), (a + b).where(long1 & (b != ""), "")]
    up = comb.str.upper()
    has_dba = up.str.contains(DBA_RE)
    if has_dba.any():
        sp = up[has_dba].str.split(DBA_RE, n=1, regex=True, expand=True).reindex(columns=[0, 1])
        parts += [sp[0].reindex(a.index).fillna(""), sp[1].reindex(a.index).fillna("")]
    out = pd.concat([pd.DataFrame({"key": key.values, "name_norm": norm_name(x).values}) for x in parts])
    return out[out.name_norm.str.len() >= 3].drop_duplicates()


def norm_city(s: pd.Series) -> pd.Series:
    s = _blank(s).map(strip_accents).str.upper()
    s = s.str.replace(r"[^A-Z0-9 ]", " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    s = s.str.replace(r"^ST\s", "SAINT ", regex=True)
    s = s.str.replace(r"^MT\s", "MOUNT ", regex=True)
    s = s.str.replace(r"^FT\s", "FORT ", regex=True)
    return s


def norm_state(s: pd.Series) -> pd.Series:
    return _blank(s).str.upper().str.replace(r"[^A-Z]", "", regex=True)


def zip5(s: pd.Series) -> pd.Series:
    raw = _blank(s)
    d = raw.str.replace(r"\D", "", regex=True)
    ok = raw.str.fullmatch(r"[\d\-\s]+") & d.str.len().isin([4, 5, 8, 9])
    d = d.where(~d.str.len().isin([4, 8]), d.str.zfill(9).where(d.str.len() == 8, d.str.zfill(5)))
    return d.str[:5].where(ok, "")


def fmt_pct(n, d):
    return round(100 * n / d, 1) if d else np.nan


# quick self-test
_t = norm_name(pd.Series(["The Women's Fund, Inc.", "BOYS & GIRLS CLUB OF BOSTON INC C/O JOHN DOE", "Foundation, The"]))
assert _t.tolist() == ["WOMENS FUND", "BOYS AND GIRLS CLUB OF BOSTON", "FOUNDATION"], _t.tolist()
assert zip5(pd.Series(["02115", "2115", "021151234", "21151234", "02115-1234", "PO BOX"])).tolist() == ["02115"] * 5 + [""]
assert norm_city(pd.Series(["St. Louis", "Mt Vernon"])).tolist() == ["SAINT LOUIS", "MOUNT VERNON"]
_c = combine_name_lines(
    pd.Series(["Massachusetts Moderators Associatio", "HABITAT FOR HUMANITY OF SALEM COUNT", "THEODORE PAYNE FDN FOR WILD FLOWERS",
               "YMCA", "THE CORNELL LAB OF ORNITHOLOGY"]),
    pd.Series(["n Inc", "HABITAT FOR HUMANITY OF SALEM COUNTY", "AND NATIVE PLANTS", "", "THE CORNELL LAB OF ORNIT"]))
assert _c.tolist() == ["Massachusetts Moderators Association Inc", "HABITAT FOR HUMANITY OF SALEM COUNTY",
                       "THEODORE PAYNE FDN FOR WILD FLOWERS AND NATIVE PLANTS", "YMCA", "THE CORNELL LAB OF ORNITHOLOGY"], _c.tolist()
_v = set(name_variants(pd.Series([1]), pd.Series(["KENNEBEC VALLEY MENTAL HEALTH CENTE"]),
                       pd.Series(["D/B/A KENNEBEC BEHAVIORAL HEALTH"])).name_norm)
assert "KENNEBEC BEHAVIORAL HEALTH" in _v, _v
print("helpers OK")

## 1. Download IRS reference files

* **EO BMF**: regional files `eo1`–`eo4` (eo4 = international and other areas) from
  https://www.irs.gov/charities-non-profits/exempt-organizations-business-master-file-extract-eo-bmf
  — every organization the IRS currently recognizes as tax-exempt. Revoked orgs and most governments are **not** in it.
* **Publication 78**: organizations eligible to receive tax-deductible contributions (pipe-delimited).
* **Auto-revocation list**: EINs whose exemption was revoked. Explains part of "valid EIN, not in BMF".

The download date is written to `reference/download_log.json` for the methodology section.

In [ ]:
BMF_URLS = {f"eo{i}.csv": f"https://www.irs.gov/pub/irs-soi/eo{i}.csv" for i in (1, 2, 3, 4)}
PUB78_URL = "https://apps.irs.gov/pub/epostcard/data-download-pub78.zip"
REVOC_URL = "https://apps.irs.gov/pub/epostcard/data-download-revocation.zip"


def download(url: str, dest: Path) -> None:
    if dest.exists() and dest.stat().st_size > 0 and not FORCE_DOWNLOAD:
        print(f"  cached  {dest.name}")
        return
    t = time.time()
    with requests.get(url, stream=True, timeout=300, headers={"User-Agent": "BU-SPARK-DS539-research"}) as r:
        r.raise_for_status()
        tmp = dest.with_suffix(dest.suffix + ".part")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
        tmp.rename(dest)
    print(f"  got     {dest.name}  {dest.stat().st_size / 1e6:.1f} MB  {time.time() - t:.0f}s")


log_path = REF_DIR / "download_log.json"
log = json.loads(log_path.read_text()) if log_path.exists() else {}
targets = {**{n: u for n, u in BMF_URLS.items()},
           "pub78.zip": PUB78_URL, "revocation.zip": REVOC_URL}
for name, url in targets.items():
    dest = REF_DIR / name
    existed = dest.exists()
    try:
        download(url, dest)
        if not existed or FORCE_DOWNLOAD:
            log[name] = {"url": url, "downloaded": date.today().isoformat()}
    except Exception as e:
        if name == "revocation.zip":
            print(f"  WARNING revocation list not downloaded ({e}); continuing without it")
        else:
            raise
log_path.write_text(json.dumps(log, indent=2))
log

## 2. Clean BMF, Pub 78 and revocation list

In [ ]:
BMF_COLS = ["EIN", "NAME", "SORT_NAME", "STREET", "CITY", "STATE", "ZIP", "SUBSECTION",
            "FOUNDATION", "DEDUCTIBILITY", "STATUS", "NTEE_CD", "RULING"]

bmf = pd.concat(
    [pd.read_csv(REF_DIR / n, dtype=str, keep_default_na=False, usecols=lambda c: c in BMF_COLS,
                 encoding="latin-1")
     for n in BMF_URLS],
    ignore_index=True,
)
bmf["EIN"] = bmf["EIN"].str.strip().str.zfill(9)
n_before = len(bmf)
bmf = bmf.drop_duplicates("EIN", keep="first")
bmf["name_norm"] = norm_name(bmf["NAME"])
bmf["sort_name_norm"] = norm_name(bmf["SORT_NAME"])
bmf["city_norm"] = norm_city(bmf["CITY"])
bmf["state"] = norm_state(bmf["STATE"])
bmf["zip5"] = zip5(bmf["ZIP"])
bmf["is_private_foundation"] = bmf["FOUNDATION"].isin(["02", "03", "04"])
print(f"BMF: {n_before:,} rows -> {len(bmf):,} unique EINs; "
      f"{bmf.is_private_foundation.sum():,} private foundations; "
      f"{(bmf.SORT_NAME.str.strip() != '').mean():.1%} have a secondary (sort) name")
bmf.to_parquet(OUT_DIR / "bmf_clean.parquet", index=False)


def read_irs_pipe_zip(path: Path) -> pd.DataFrame:
    """IRS 'epostcard' downloads: one pipe-delimited .txt inside a zip, header row optional."""
    with zipfile.ZipFile(path) as z:
        member = [m for m in z.namelist() if m.lower().endswith(".txt")][0]
        text = z.read(member).decode("latin-1")
    rows = list(csv.reader(io.StringIO(text), delimiter="|", quoting=csv.QUOTE_NONE))
    rows = [r for r in rows if r and any(x.strip() for x in r)]
    if not re.fullmatch(r"\d{1,9}", rows[0][0].strip()):
        rows = rows[1:]  # header row
    width = max(len(r) for r in rows[:1000])
    return pd.DataFrame([r + [""] * (width - len(r)) for r in rows]).iloc[:, :width]


p78 = read_irs_pipe_zip(REF_DIR / "pub78.zip")
p78 = p78.iloc[:, :6]
p78.columns = ["EIN", "NAME", "CITY", "STATE", "COUNTRY", "DEDUCTIBILITY_CODE"][: p78.shape[1]]
p78["EIN"] = p78["EIN"].str.strip().str.zfill(9)
p78 = p78.drop_duplicates("EIN")
p78["name_norm"] = norm_name(p78["NAME"])
p78["city_norm"] = norm_city(p78["CITY"])
p78["state"] = norm_state(p78["STATE"])
print(f"Pub 78: {len(p78):,} EINs; {(~p78.EIN.isin(bmf.EIN)).sum():,} are not in the BMF")
p78.to_parquet(OUT_DIR / "pub78_clean.parquet", index=False)

if (REF_DIR / "revocation.zip").exists():
    rev = read_irs_pipe_zip(REF_DIR / "revocation.zip")
    rev = rev.iloc[:, :2]
    rev.columns = ["EIN", "NAME"]
    rev["EIN"] = rev["EIN"].str.strip().str.zfill(9)
    rev = rev.drop_duplicates("EIN")
    rev["name_norm"] = norm_name(rev["NAME"])
    print(f"Revocation list: {len(rev):,} EINs; {rev.EIN.isin(bmf.EIN).sum():,} also in BMF (reinstated)")
else:
    rev = pd.DataFrame(columns=["EIN", "NAME", "name_norm"])

## 3. Load Pharos tables

Everything is read as text (EINs keep their leading zeros). Column names follow the team data dictionary;
they are renamed to short names for readability.

In [ ]:
PF_GRANT_COLS = {
    "ein": "filer_ein",
    "taxyear": "taxyear",
    "line_number": "line_number",
    "rcpnt_ein": "ein_raw",
    "rcpntbsnssnm_bsnssnmln1txt": "rcp_name1",
    "rcpntbsnssnm_bsnssnmln2txt": "rcp_name2",
    "grntorcntrbtnpddryr_rcpntprsnnm": "rcp_person",
    "rcpntusaddrss_addrssln1txt": "rcp_addr1",
    "rcpntusaddrss_ctynm": "rcp_city",
    "rcpntusaddrss_sttabbrvtncd": "rcp_state",
    "rcpntusaddrss_zipcd": "rcp_zip",
    "rcpntfrgnaddrss_cntrycd": "rcp_country",
    "grntorcntrbtnpddryr_rcpntfndtnsttstxt": "rcp_status_txt",
    "grntorcntrbtnpddryr_amt": "amount",
    "grntorcntrbtnpddryr_grntorcntrbtnprpstxt": "purpose",
}


def read_cols(path: Path, mapping: dict) -> pd.DataFrame:
    head = pd.read_csv(path, nrows=0).columns
    missing = [c for c in mapping if c not in head]
    assert not missing, f"{path.name} is missing columns {missing}"
    return pd.read_csv(path, dtype=str, keep_default_na=False, usecols=list(mapping)).rename(columns=mapping)


t0 = time.time()
g = read_cols(PATHS["pf_grants"], PF_GRANT_COLS)
g["filer_ein"] = g["filer_ein"].str.strip().str.zfill(9)
g["amount"] = pd.to_numeric(g["amount"].str.replace(",", ""), errors="coerce").fillna(0)
print(f"PF grants: {len(g):,} rows, {g.filer_ein.nunique():,} foundations, "
      f"${g.amount.sum() / 1e9:,.1f}B  ({time.time() - t0:.0f}s)")

pf_h = read_cols(PATHS["pf_header"], {
    "ein": "ein", "taxyear": "taxyear", "nameoffoundation": "name",
    "cityortown": "city", "stateorprovince": "state", "zipcode": "zip"})
pf_h["ein"] = pf_h["ein"].str.strip().str.zfill(9)

f990_h = read_cols(PATHS["f990_header"], {
    "f9_00_org_ein": "ein", "f9_00_tax_year": "taxyear",
    "f9_00_org_name_l1": "name1", "f9_00_org_name_l2": "name2", "f9_00_org_name_dba_l1": "dba",
    "f9_00_org_addr_l1": "street", "f9_00_org_addr_city": "city",
    "f9_00_org_addr_state": "state", "f9_00_org_addr_zip": "zip"})
f990_h["ein"] = f990_h["ein"].str.strip().str.zfill(9)
print(f"990 header: {len(f990_h):,} rows / {f990_h.ein.nunique():,} EINs;  "
      f"PF header: {len(pf_h):,} rows / {pf_h.ein.nunique():,} EINs")

# Scope filters
if FILER_STATES:
    keep = set(pf_h.loc[norm_state(pf_h.state).isin(FILER_STATES), "ein"])
    g = g[g.filer_ein.isin(keep)]
    print(f"FILER_STATES={FILER_STATES}: {len(g):,} grant rows kept")
if RECIPIENT_STATES:
    g = g[norm_state(g.rcp_state).isin(RECIPIENT_STATES)]
    print(f"RECIPIENT_STATES={RECIPIENT_STATES}: {len(g):,} grant rows kept")
g = g.reset_index(drop=True)
g["row_id"] = np.arange(len(g))

# filer state, for the report slice
filer_state = (pf_h.assign(state=norm_state(pf_h.state)).sort_values("taxyear")
               .drop_duplicates("ein", keep="last").set_index("ein")["state"])
g["filer_state"] = g.filer_ein.map(filer_state).fillna("")

## 4. Reference index

One long table `(ein, name_norm, city_norm, state, zip5, source)` with every name we know for every EIN:
BMF legal name and secondary name, Pub 78 name, 990 header name (every reading of its two lines, see
`name_variants`) and DBA, and 990-PF foundation name.
Our own filers are included because many recipients file a 990 themselves, and the filer address is the most
recent one we have. This table is also saved for task 4.

In [ ]:
def ref_variants(df, l1, l2, source):
    v = name_variants(pd.Series(df.index), df[l1], df[l2])
    v = v.merge(df[["ein", "city", "state", "zip"]], left_on="key", right_index=True)
    return pd.DataFrame({"ein": v.ein.values, "name_norm": v.name_norm.values, "city_norm": norm_city(v.city).values,
                         "state": norm_state(v.state).values, "zip5": zip5(v.zip).values, "source": source})


def ref_part(ein, name, city, state, zipc, source):
    return pd.DataFrame({
        "ein": ein.values, "name_norm": norm_name(name).values, "city_norm": norm_city(city).values,
        "state": norm_state(state).values, "zip5": zip5(zipc).values if zipc is not None else "",
        "source": source})


f990_u = f990_h.sort_values("taxyear").drop_duplicates(["ein", "name1", "name2", "dba", "zip"], keep="last")
pf_u = pf_h.sort_values("taxyear").drop_duplicates(["ein", "name", "zip"], keep="last")

ref = pd.concat([
    ref_part(bmf.EIN, bmf.NAME, bmf.CITY, bmf.STATE, bmf.ZIP, "bmf"),
    ref_part(bmf.EIN, bmf.SORT_NAME, bmf.CITY, bmf.STATE, bmf.ZIP, "bmf_sort_name"),
    ref_part(p78.EIN, p78.NAME, p78.CITY, p78.STATE, None, "pub78"),
    ref_variants(f990_u.reset_index(drop=True), "name1", "name2", "990"),
    ref_part(f990_u.ein, f990_u.dba, f990_u.city, f990_u.state, f990_u.zip, "990_dba"),
    ref_part(pf_u.ein, pf_u.name, pf_u.city, pf_u.state, pf_u.zip, "990pf"),
], ignore_index=True)
ref = ref[ref.name_norm.str.len() >= 3].drop_duplicates(["ein", "name_norm", "city_norm", "state", "zip5"])
print(f"Reference index: {len(ref):,} rows, {ref.ein.nunique():,} EINs")
print(ref.source.value_counts().to_string())
ref.to_parquet(OUT_DIR / "reference_name_index.parquet", index=False)

EIN_SETS = {
    "bmf": set(bmf.EIN), "pub78": set(p78.EIN), "990": set(f990_h.ein),
    "990pf": set(pf_h.ein), "revoked": set(rev.EIN),
}

## 5. Clean the reported recipient EINs

`ein_form` describes what was typed in `rcpnt_ein`:

| ein_form | meaning |
|---|---|
| `valid9` | 9 digits (hyphens and spaces removed) |
| `padded8` | 8 digits, probably lost a leading zero; left-padded |
| `word_INDIVIDUAL` / `word_FOREIGN` / `word_GOVERNMENT` | the filer typed the word instead of an EIN |
| `placeholder` | 000000000, 111111111, 123456789, ... |
| `blank` | empty |
| `other` | anything else |

In [ ]:
raw = _blank(g.ein_raw).str.upper()
digits = raw.str.replace(r"[\s\-]", "", regex=True)
is_num = digits.str.fullmatch(r"\d+")
word = raw.str.replace(r"[^A-Z]", "", regex=True)

g["ein_form"] = np.select(
    [raw == "",
     word.isin(["INDIVIDUAL", "INDIVIDUALS", "IND"]),
     word.isin(["FOREIGN", "FOREIGNORG", "FOREIGNORGANIZATION"]),
     word.isin(["GOVERNMENT", "GOVT", "GOV", "GOVERNMENTAL"]),
     is_num & (digits.str.len() == 9),
     is_num & (digits.str.len() == 8)],
    ["blank", "word_INDIVIDUAL", "word_FOREIGN", "word_GOVERNMENT", "valid9", "padded8"],
    "other")
g["ein_clean"] = np.where(g.ein_form.isin(["valid9", "padded8"]), digits.str.zfill(9), "")
placeholder = g.ein_clean.str.fullmatch(r"(\d)\1{8}") | g.ein_clean.isin(["123456789", "987654321"])
g.loc[placeholder, "ein_form"] = "placeholder"
g.loc[placeholder, "ein_clean"] = ""

print(g.ein_form.value_counts(normalize=True).mul(100).round(2).to_string())

## 6. Classify recipients

First matching rule wins. Note that a name in the *person* field is not enough to call a recipient an individual:
filers often type organization names there, so it only counts when there is no EIN, the status text does not
say PC / NC / PF / 501(c)(3) / GOV etc., and the name has no organization word (SOCIETY, CHURCH, SCHOOL, ...).

Name-pattern governments (`CITY OF ...`, `TOWN OF ...`, `... SCHOOL DISTRICT`) are only
a *flag* at this point: names like *City of Hope* are nonprofits, so the flag becomes the class only if the row
does not match an organization later (step 8).

In [ ]:
# The recipient's name can sit in the business-name lines OR in the person-name field: many filers type
# organization names into the person field (e.g. ~144k rows with a valid 9-digit EIN and only a "person" name such
# as MASSACHUSETTS GENERAL HOSPITAL). So the person field alone does NOT make a recipient an individual.
has_org_field = (_blank(g.rcp_name1) != "") | (_blank(g.rcp_name2) != "")
has_person = _blank(g.rcp_person) != ""
person_only = has_person & ~has_org_field
g["name_from_person_field"] = person_only
g["rcp_l1"] = np.where(person_only, _blank(g.rcp_person), _blank(g.rcp_name1))
g["rcp_l2"] = np.where(person_only, "", _blank(g.rcp_name2))
g["rcp_name_full"] = combine_name_lines(g.rcp_l1, g.rcp_l2)
g["rcp_name_norm"] = norm_name(g.rcp_name_full)
# line 1 hit a software length limit with nothing on line 2 to complete it -> task 4 should try prefix matching
g["name_maybe_truncated"] = (g.rcp_l2 == "") & g.rcp_l1.str.len().isin([35, 75])
has_name = g.rcp_name_norm != ""
country = _blank(g.rcp_country).str.upper()
status = _blank(g.rcp_status_txt).str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
g["rcp_state_n"] = norm_state(g.rcp_state)
g["rcp_city_n"] = norm_city(g.rcp_city)
g["rcp_zip5"] = zip5(g.rcp_zip)
has_valid_ein = g.ein_clean != ""

# Status text is free text, but most of it comes from tax-software pick lists:
# PC, NC, PF, POF, EOF, SO I, 501(C)3, PUBLIC CHARITY, GOV, GOV: FOREIGN GOVERNM, I, I Individual person ...
IND_STATUS_RE = r"^(I|IND|INDIV\w*|INDIVDUAL\w*|IINDIVIDUAL\w*|INDIVIDUAL\w*|PERSON|STUDENT\w*|SCHOLARSHIP\w*)$"
GOV_STATUS_RE = r"^(GOV|LOCALGOV|STATEGOV|MUNICIPAL|PUBLICAGENCY|GU$)|GOVERNMENT|170C1"
ORG_STATUS_RE = (r"^(PC|PF|POF|NC|EOF|SO|501|509|PUB|EXEMPT|TAXEXEMPT|CHARIT|NONPROFIT|OTHERPUB|CHURCH|SCHOOL|EDUC|"
                 r"HOSPITAL|UNIV|COLLEGE)")
FOREIGN_STATUS_RE = r"^(FOREIGN|FORGN|NONUS)"
# "SEE ATTACHED ..." at the start; placeholder words only when they are the whole name
UNSPECIFIED_RE = (r"^(SEE (ATTACHED|ATTACHMENT|STATEMENT|SCHEDULE|LIST|EXHIBIT)|(ATTACHED|LIST|SCHEDULE|STATEMENT) "
                  r"(LIST|SCHEDULE|STATEMENT|ATTACHED)|DETAILS OF (CONTRIBUTIONS|GRANTS|DONATIONS))\b"
                  r"|\bAVAILABLE UPON REQUEST\b"
                  r"|^(VARIOUS|MULTIPLE|MISC|MISCELLANEOUS|N A|NA|NONE|UNKNOWN|ANONYMOUS|ATTACHED)"
                  r"( (RECIPIENTS|ORGANIZATIONS|ORGS|CHARITIES|CHARITABLE ORGANIZATIONS|GRANTEES|DONEES))?$")
GOV_NAME_RE = (r"^(CITY|TOWN|VILLAGE|COUNTY|BOROUGH|TOWNSHIP|STATE|COMMONWEALTH|PARISH|CITY AND COUNTY) OF\b"
               r"|\b(SCHOOL DISTRICT|PUBLIC SCHOOLS|BOARD OF EDUCATION|SCHOOL DEPARTMENT|POLICE DEPARTMENT|"
               r"PUBLIC LIBRARY|DEPARTMENT OF|UNITED STATES)\b|\bISD$")
# words that make a name in the person field read as an organization
ORG_WORD_RE = (r"\b(INC|CORP|FOUNDATION|FDN|FUND|TRUST|SOCIETY|ASSOCIATION|ASSN|ASSOC|CLUB|CHURCH|PARISH|TEMPLE|"
               r"SYNAGOGUE|MINISTRY|MINISTRIES|MISSION|SCHOOL|ACADEMY|UNIVERSITY|UNIV|COLLEGE|INSTITUTE|CENTER|CENTRE|"
               r"CTR|MUSEUM|LIBRARY|HOSPITAL|CLINIC|HEALTH|SERVICES|COUNCIL|COMMITTEE|LEAGUE|ALLIANCE|COALITION|"
               r"NETWORK|PROJECT|PROGRAM|SHELTER|RESCUE|FOOD BANK|PANTRY|ORCHESTRA|SYMPHONY|THEATER|THEATRE|"
               r"OPERA|BALLET|CHOIR|CHORUS|AUTHORITY|DEPARTMENT|DEPT|COMMISSION|BOARD|FRIENDS OF|CHAPTER|LODGE|POST|"
               r"YMCA|YWCA|SCOUTS|UNITED WAY|HABITAT|SALVATION ARMY|RED CROSS|COMMUNITY|INTERNATIONAL|NATIONAL|"
               r"AMERICAN|PARTNERS|COOPERATIVE|CO OP|SANCTUARY|PRESERVE|CONSERVANCY|RESEARCH|PTA|PTO|BOOSTERS?)\b")
status_org = status.str.contains(ORG_STATUS_RE)
looks_like_org = g.rcp_name_norm.str.contains(ORG_WORD_RE)
status_gov = status.str.contains(GOV_STATUS_RE)

rules = [
    ("foreign",        "ein_word_FOREIGN",        g.ein_form == "word_FOREIGN"),
    ("foreign",        "foreign_country_code",    (country != "") & ~country.isin(["US", "USA"])),
    ("foreign",        "status_text_foreign",     status.str.contains(FOREIGN_STATUS_RE) & ~has_valid_ein),
    ("foreign",        "non_us_state",            (g.rcp_state_n.str.len() == 2) & ~g.rcp_state_n.isin(US_STATE_CODES) & ~has_valid_ein),
    ("individual",     "ein_word_INDIVIDUAL",     g.ein_form == "word_INDIVIDUAL"),
    ("individual",     "status_text_individual",  status.str.contains(IND_STATUS_RE) & ~has_valid_ein),
    ("individual",     "person_field_no_ein",     person_only & ~has_valid_ein & ~status_org & ~status_gov & ~looks_like_org),
    ("government",     "ein_word_GOVERNMENT",     g.ein_form == "word_GOVERNMENT"),
    ("government",     "status_text_government",  status_gov & ~has_valid_ein),
    ("unspecified",    "see_attached_various",    g.rcp_name_norm.str.contains(UNSPECIFIED_RE) & ~has_valid_ein),
    ("unidentifiable", "no_name_no_ein",          ~has_name & ~has_valid_ein),
]
g["recipient_class"] = np.select([c for _, _, c in rules], [k for k, _, _ in rules], "organization")
g["class_reason"] = np.select([c for _, _, c in rules], [r for _, r, _ in rules], "")
g["gov_name_flag"] = g.rcp_name_norm.str.contains(GOV_NAME_RE)

print(pd.crosstab(g.class_reason.replace("", "(default organization)"), g.recipient_class, margins=True).to_string())
print(f"\nName taken from the person-name field: {g.name_from_person_field.sum():,} rows, of which "
      f"{(g.name_from_person_field & (g.recipient_class == 'organization')).sum():,} are kept as organizations")
print(f"Names that look truncated (35/75 chars, no line 2): {g.name_maybe_truncated.sum():,}")

## 7. Verify the reported EINs

For every row with a usable EIN: which reference sources contain it, and does the recipient name agree with
any name on file for that EIN (`token_set_ratio`, 0–100, best over all names). A low score is only a flag —
acronyms (`YMCA` vs `YOUNG MENS CHRISTIAN ASSOCIATION`) score low but are fine.

| ein_status | meaning |
|---|---|
| `self_ein` | recipient EIN = the foundation's own EIN (data entry error) |
| `found_bmf` | in the BMF |
| `found_pub78` | not in BMF, in Pub 78 |
| `found_990_filer` | not in BMF/Pub 78, but files a 990 or 990-PF in our data |
| `revoked_only` | only on the revocation list (an org that lost exempt status). Its name on the list is compared too: a mismatch means the filer typed someone else's EIN |
| `not_found` | nowhere |

In [ ]:
e = g.ein_clean
for k, s in EIN_SETS.items():
    g[f"ein_in_{k}"] = e.isin(s) & (e != "")

g["ein_status"] = np.select(
    [e == "",
     e == g.filer_ein,
     g.ein_in_bmf,
     g.ein_in_pub78,
     g.ein_in_990 | g.ein_in_990pf,
     g.ein_in_revoked],
    ["", "self_ein", "found_bmf", "found_pub78", "found_990_filer", "revoked_only"],
    "not_found")

# name agreement: found EINs against every name on file; revoked-only EINs against the revocation list's name
chk = g.loc[(g.ein_status.str.startswith("found") | (g.ein_status == "revoked_only")) & (g.rcp_name_norm != ""),
            ["row_id", "ein_clean", "rcp_name_norm"]]
ref_names = pd.concat([ref[["ein", "name_norm"]],
                       rev.loc[rev.name_norm.str.len() >= 3, ["EIN", "name_norm"]].rename(columns={"EIN": "ein"})]
                      ).drop_duplicates()
pairs = chk.merge(ref_names, left_on="ein_clean", right_on="ein", how="inner")
try:
    from rapidfuzz.process import cpdist
    pairs["score"] = cpdist(pairs.rcp_name_norm.tolist(), pairs.name_norm.tolist(),
                            scorer=fuzz.token_set_ratio, workers=-1)
except ImportError:
    pairs["score"] = [fuzz.token_set_ratio(a, b) for a, b in zip(pairs.rcp_name_norm, pairs.name_norm)]
best = pairs.groupby("row_id")["score"].max()
g["ein_name_score"] = g.row_id.map(best)
print(f"Name-agreement pairs scored: {len(pairs):,}")
print(g.loc[g.ein_status != "", "ein_status"].value_counts().to_string())
print("Share of found EINs whose name disagrees (score < threshold): "
      f"{(g.ein_name_score < NAME_AGREE_THRESHOLD).sum() / g.ein_status.str.startswith('found').sum():.1%}")

## 8. Exact matching on name + ZIP5, then name + city + state

Candidates: organization/government rows with a recipient name and either no usable EIN, or an EIN that is
`self_ein` / `not_found` / `revoked_only`, or an EIN whose name disagrees.

Each row is tried with every reading of its two name lines (`name_variants`: line 1, line 2, combined, joined
without a space, both sides of a D/B/A), because filers put the real name on either line and line 1 is often
cut at 35 characters. A row counts as matched only when **exactly one EIN** comes back;
2+ EINs is `ambiguous` and the candidates are kept for task 4.

In [ ]:
BAD_EIN = g.ein_status.isin(["self_ein", "not_found", "revoked_only"])
NAME_DISAGREES = g.ein_status.str.startswith("found") & (g.ein_name_score < NAME_AGREE_THRESHOLD)
cand_mask = (g.recipient_class.isin(["organization", "government"]) & (g.rcp_name_norm != "")
             & ((g.ein_clean == "") | BAD_EIN | NAME_DISAGREES))
cand = g.loc[cand_mask, ["row_id", "rcp_city_n", "rcp_state_n", "rcp_zip5"]]

variants = name_variants(g.row_id[cand_mask], g.rcp_l1[cand_mask], g.rcp_l2[cand_mask]).rename(columns={"key": "row_id"})
variants = variants.merge(cand, on="row_id")
print(f"Exact-match candidates: {cand.shape[0]:,} rows, {len(variants):,} name variants")


def exact_pass(v: pd.DataFrame, keys_left: list, keys_right: list, label: str) -> pd.DataFrame:
    r = ref.loc[ref[keys_right[-1]] != "", ["ein"] + keys_right].drop_duplicates()
    m = v.merge(r, left_on=["name_norm"] + keys_left, right_on=["name_norm"] + keys_right[1:], how="inner")
    out = m.groupby("row_id")["ein"].agg(n_candidates="nunique", candidates=lambda s: "|".join(sorted(set(s))[:5]))
    out["pass"] = label
    return out.reset_index()


p1 = exact_pass(variants[variants.rcp_zip5 != ""], ["rcp_zip5"], ["name_norm", "zip5"], "name_zip")
left = variants[~variants.row_id.isin(p1.loc[p1.n_candidates == 1, "row_id"])]
p2 = exact_pass(left[(left.rcp_city_n != "") & (left.rcp_state_n != "")],
                ["rcp_city_n", "rcp_state_n"], ["name_norm", "city_norm", "state"], "name_city")

res = pd.concat([p1[p1.n_candidates == 1], p2[p2.n_candidates == 1]]).drop_duplicates("row_id")
amb = pd.concat([p1[p1.n_candidates > 1], p2[p2.n_candidates > 1]])
amb = amb[~amb.row_id.isin(res.row_id)].sort_values("n_candidates").drop_duplicates("row_id")
print(f"name+ZIP unique: {(p1.n_candidates == 1).sum():,}   name+city unique: {(p2.n_candidates == 1).sum():,}   "
      f"ambiguous: {len(amb):,}")

## 9. Final match columns

`match_method` (one per row):

| match_method | matched? | meaning |
|---|---|---|
| `ein_verified` | yes | reported EIN exists and the name agrees |
| `ein_no_name` | yes | reported EIN exists, no recipient name to check it against |
| `ein_name_disagrees` | yes, flagged | reported EIN is in a reference but the name disagrees (often acronyms or sub-units, e.g. NYU School of Medicine under NYU Langone). Never overwritten; any other exact-match EIN goes to `alt_ein` |
| `ein_government_likely` | yes, flagged | reported EIN not in any reference but has the XX-60xxxxx shape of a state/local government unit (state universities, school districts); kept as reported |
| `ein_revoked` | yes, flagged | reported EIN is only on the revocation list **and** the recipient name agrees with the name there. A disagreeing name means the filer typed another org's EIN: the row is treated as having no EIN |
| `ein_self_grant` | yes, flagged | recipient EIN and name are the foundation's own (e.g. its own scholarship fund) |
| `ein_corrected_name_zip` / `ein_corrected_name_city` | yes | reported EIN was the foundation's own, or revoked with a disagreeing name, or not found / revoked and within 2 digits of the EIN an exact name match found (a typo). Never for government recipients |
| `ein_unlisted_kept` | yes, flagged | reported 9-digit EIN is in no IRS list, and the exact name match points to an unrelated EIN. The reported EIN is kept (spot check: Ohio Univ., Oregon State, Oklahoma State, Delbarton were all right); the other EIN goes to `alt_ein` |
| `name_zip` / `name_city` | yes | no usable EIN; exact name match found one EIN |
| `gov_name_review` | no → review | filer marked the recipient as government and an exact name match found one EIN. Governments are mostly not in the BMF, so that EIN is often a different same-name org (spot check: City of Olympia). Candidate kept in `alt_ein` |
| `ambiguous` | no → task 4 | exact name match found 2+ EINs |
| `unmatched` | no → task 4 | nothing found |
| `set_aside` | excluded | individual, foreign, unspecified or unidentifiable |

In [ ]:
g = g.merge(res[["row_id", "candidates", "pass"]].rename(columns={"candidates": "exact_ein", "pass": "exact_pass"}),
            on="row_id", how="left")
g = g.merge(amb[["row_id", "n_candidates", "candidates"]].rename(columns={"candidates": "ambiguous_candidates"}),
            on="row_id", how="left")
g[["exact_ein", "exact_pass", "ambiguous_candidates"]] = g[["exact_ein", "exact_pass", "ambiguous_candidates"]].fillna("")

found = g.ein_status.str.startswith("found")
exact = g.exact_ein != ""
set_aside = ~g.recipient_class.isin(["organization", "government"])
is_self = g.ein_status == "self_ein"
not_found = g.ein_status == "not_found"
revoked = g.ein_status == "revoked_only"


def digit_diff(a: pd.Series, b: pd.Series) -> pd.Series:
    """Number of positions where two 9-digit EINs differ (99 if either is missing). A typo or a swap of two
    neighbouring digits gives 1-2; an unrelated EIN gives ~7-9."""
    return pd.Series([sum(x != y for x, y in zip(u, v)) if len(u) == 9 and len(v) == 9 else 99
                      for u, v in zip(a, b)], index=a.index)


g["ein_digit_diff"] = digit_diff(g.ein_clean, g.exact_ein)
close = g.ein_digit_diff <= 2
# State and local government units (state universities, school districts, cities) have real EINs but are not in
# the BMF. In this data their EINs look like XX-60xxxxx (Penn State 24-6000376, Univ. of Washington 91-6001029).
# Such an EIN is kept as reported rather than "corrected" to a same-name BMF entity (alumni association, foundation).
gov_ein = not_found & (g.ein_clean.str[2:4] == "60")

# Fix from the spot check (GBH, University of Montana): a revoked-only EIN counts only if the recipient name agrees
# with the name on the revocation list. Otherwise the filer typed another org's EIN, and it is treated as no EIN.
revoked_ok = revoked & ~(g.ein_name_score < NAME_AGREE_THRESHOLD)
revoked_bad = revoked & (g.ein_name_score < NAME_AGREE_THRESHOLD)
# Fix from the spot check (City of Olympia, Univ. of Southern Indiana): recipients the filer marked as government are
# mostly not in the BMF, so a unique same-name BMF entity is often a different org. Keep it as a candidate only.
gov_recipient = g.recipient_class == "government"

g["match_method"] = np.select(
    [set_aside,
     found & exact & (g.exact_ein == g.ein_clean),
     is_self & exact & (g.exact_ein == g.ein_clean),       # the foundation names itself as recipient
     found & (g.rcp_name_norm == ""),
     found & (g.ein_name_score >= NAME_AGREE_THRESHOLD),
     found & (g.ein_name_score < NAME_AGREE_THRESHOLD),   # a BMF-listed EIN is never overwritten
     exact & ~gov_recipient & (is_self | revoked_bad | ((not_found | revoked) & close)) & (g.exact_pass == "name_zip"),
     exact & ~gov_recipient & (is_self | revoked_bad | ((not_found | revoked) & close)) & (g.exact_pass == "name_city"),
     gov_ein,
     revoked_ok,
     # Fix from the spot check (Ohio Univ., Oregon State, Oklahoma State, Delbarton): a 9-digit EIN that is not in
     # any IRS list but was reported by the filer is kept, even when the name points to a different, unrelated EIN.
     not_found & exact,
     (g.ein_clean == "") | revoked_bad | is_self,           # placeholder; refined just below
     g.ambiguous_candidates != ""],
    ["set_aside", "ein_verified", "ein_self_grant", "ein_no_name", "ein_verified", "ein_name_disagrees",
     "ein_corrected_name_zip", "ein_corrected_name_city", "ein_government_likely", "ein_revoked",
     "ein_unlisted_kept", "_no_usable_ein", "ambiguous"],
    "unmatched")
no_ein = g.match_method == "_no_usable_ein"
g["match_method"] = np.select(
    [no_ein & exact & gov_recipient,
     no_ein & exact & (g.exact_pass == "name_zip"),
     no_ein & exact & (g.exact_pass == "name_city"),
     no_ein & (g.ambiguous_candidates != ""),
     no_ein],
    ["gov_name_review", "name_zip", "name_city", "ambiguous", "unmatched"],
    g.match_method)
# found EIN whose recipient name exists but has no reference name to compare (score NaN) -> verified by existence
nan_score = found & (g.rcp_name_norm != "") & g.ein_name_score.isna() & (g.match_method == "unmatched")
g.loc[nan_score, "match_method"] = "ein_no_name"

KEEP_REPORTED = ["ein_verified", "ein_self_grant", "ein_no_name", "ein_name_disagrees", "ein_revoked",
                 "ein_government_likely", "ein_unlisted_kept"]
USE_EXACT = ["ein_corrected_name_zip", "ein_corrected_name_city", "name_zip", "name_city"]
g["matched_ein"] = np.select([g.match_method.isin(KEEP_REPORTED), g.match_method.isin(USE_EXACT)],
                             [g.ein_clean, g.exact_ein], "")
# the other EIN an exact name match pointed to, kept for review when it was not used
g["alt_ein"] = np.where(g.match_method.isin(KEEP_REPORTED + ["gov_name_review"]) & exact & (g.exact_ein != g.ein_clean),
                        g.exact_ein, "")
g["is_matched"] = g.matched_ein != ""

# Name-pattern governments that found no organization become class "government"
gov_by_ein = (g.match_method == "ein_government_likely") & (g.recipient_class == "organization")
g.loc[gov_by_ein, "recipient_class"] = "government"
g.loc[gov_by_ein, "class_reason"] = "government_ein_not_in_bmf"
gov_by_name = g.gov_name_flag & ~g.is_matched & (g.recipient_class == "organization")
g.loc[gov_by_name, "recipient_class"] = "government"
g.loc[gov_by_name, "class_reason"] = "name_pattern"

# Matched recipient's own address: BMF first, then latest 990 / 990-PF header, then Pub 78 (city/state only)
addr = pd.concat([
    bmf[["EIN", "STREET", "CITY", "STATE", "ZIP"]].set_axis(["ein", "street", "city", "state", "zip"], axis=1).assign(src="bmf"),
    f990_h.sort_values("taxyear", ascending=False)[["ein", "street", "city", "state", "zip"]].assign(src="990"),
    pf_h.sort_values("taxyear", ascending=False)[["ein", "city", "state", "zip"]].assign(street="", src="990pf"),
    p78[["EIN", "CITY", "STATE"]].set_axis(["ein", "city", "state"], axis=1).assign(street="", zip="", src="pub78"),
]).drop_duplicates("ein")
addr.columns = ["matched_" + c if c != "ein" else c for c in addr.columns]
addr = addr.rename(columns={"matched_src": "matched_addr_source"})
addr["matched_zip5"] = zip5(addr.matched_zip)
g = g.merge(addr.drop(columns="matched_zip"), left_on="matched_ein", right_on="ein", how="left").drop(columns="ein")
g["matched_addr_source"] = g.matched_addr_source.fillna("")

# MA flag for lining up with tasks 1 and 2. Reported recipient state first; when it is blank, the ZIP
# (MA ZIP3 = 010-027 and 055), then the matched organization's own address.
MA_ZIP3 = {f"{i:03d}" for i in range(10, 28)} | {"055"}
matched_state_n = norm_state(g.matched_state)
g["recipient_in_ma"] = np.select(
    [g.rcp_state_n != "", g.rcp_zip5 != "", matched_state_n != ""],
    [g.rcp_state_n == "MA", g.rcp_zip5.str[:3].isin(MA_ZIP3), matched_state_n == "MA"],
    False).astype(bool)
g["ma_flag_source"] = np.select(
    [~g.recipient_in_ma, g.rcp_state_n != "", g.rcp_zip5 != ""],
    ["", "reported_state", "reported_zip"], "matched_address")
print(f"recipient_in_ma: {g.recipient_in_ma.sum():,} rows "
      f"({g.recipient_in_ma.mean():.1%}), ${g.amount[g.recipient_in_ma].sum() / 1e6:,.0f}M")
print(g.loc[g.recipient_in_ma, "ma_flag_source"].value_counts().to_string())
print(g.match_method.value_counts().to_string())

## 10. Match-rate table

Reported for three slices: **overall** (national), **recipient_in_ma** (lines up with tasks 1 and 2) and
**filer_in_ma** (grants made by MA foundations).

Three denominators, rows and dollars:

* **All rows** — directly comparable with Curt's ~80% (EIN filled in 80.6% of rows, 72.4% a 9-digit EIN).
* **Excl. individual / foreign / unspecified** — the rate the team reports (Joins sheet).
* **Organizations only** — also excluding governments, which mostly have no BMF entry.

Report the **row** rates first. Dollar rates are dominated by a handful of rows: e.g. pharmaceutical
patient-assistance foundations report billions in donated drugs as single "INDIVIDUAL" grants
(the top 10 individual grants are ~$15B of the ~$16B to individuals).

In [ ]:
METHOD_ORDER = ["ein_verified", "ein_self_grant", "ein_no_name", "ein_name_disagrees", "ein_government_likely",
                "ein_unlisted_kept", "ein_revoked",
                "ein_corrected_name_zip", "ein_corrected_name_city", "name_zip", "name_city",
                "gov_name_review", "ambiguous", "unmatched", "set_aside"]


def rate_table(df: pd.DataFrame) -> pd.DataFrame:
    denoms = {
        "all_rows": pd.Series(True, index=df.index),
        "excl_set_aside": df.match_method != "set_aside",
        "orgs_only": (df.match_method != "set_aside") & (df.recipient_class == "organization"),
    }
    out = []
    for m in METHOD_ORDER:
        mm = df.match_method == m
        row = {"match_method": m, "rows": int(mm.sum()), "dollars": df.amount[mm].sum()}
        for dn, dm in denoms.items():
            row[f"pct_rows_{dn}"] = fmt_pct((mm & dm).sum(), dm.sum()) if (m != "set_aside" or dn == "all_rows") else np.nan
            row[f"pct_$_{dn}"] = fmt_pct(df.amount[mm & dm].sum(), df.amount[dm].sum()) if (m != "set_aside" or dn == "all_rows") else np.nan
        out.append(row)
    t = pd.DataFrame(out)
    tot = {"match_method": "TOTAL MATCHED", "rows": int(df.is_matched.sum()), "dollars": df.amount[df.is_matched].sum()}
    for dn, dm in denoms.items():
        tot[f"pct_rows_{dn}"] = fmt_pct((df.is_matched & dm).sum(), dm.sum())
        tot[f"pct_$_{dn}"] = fmt_pct(df.amount[df.is_matched & dm].sum(), df.amount[dm].sum())
    return pd.concat([t, pd.DataFrame([tot])], ignore_index=True)


slices = {
    "overall": g,
    "recipient_in_ma": g[g.recipient_in_ma],
    "filer_in_ma": g[g.filer_state == "MA"],  # grants made by MA foundations, wherever the recipient is
}
tables = []
for name, df in slices.items():
    t = rate_table(df)
    t.insert(0, "slice", name)
    tables.append(t)
    print(f"\n=== {name}: {len(df):,} grant rows, ${df.amount.sum() / 1e6:,.0f}M ===")
    print(t.drop(columns="slice").to_string(index=False))
match_rates = pd.concat(tables, ignore_index=True)

# Benchmarks from the raw EIN column, for the comparison with Curt's ~80%
bench = pd.DataFrame([
    {"metric": "rcpnt_ein filled (any value)", "pct_rows": fmt_pct((g.ein_form != "blank").sum(), len(g))},
    {"metric": "rcpnt_ein is a usable EIN (valid9/padded8)", "pct_rows": fmt_pct((g.ein_clean != "").sum(), len(g))},
    {"metric": "usable EIN that exists in a reference", "pct_rows": fmt_pct(found.sum(), len(g))},
    {"metric": "matched after task 3 (all rows)", "pct_rows": fmt_pct(g.is_matched.sum(), len(g))},
    {"metric": "matched after task 3 (excl. set-aside)",
     "pct_rows": fmt_pct(g.is_matched.sum(), (g.match_method != "set_aside").sum())},
    {"metric": "MA recipients: matched (excl. set-aside)",
     "pct_rows": fmt_pct((g.is_matched & g.recipient_in_ma).sum(),
                         ((g.match_method != "set_aside") & g.recipient_in_ma).sum())},
])
print("\n", bench.to_string(index=False))

# Class breakdown
cls = (g.groupby(["recipient_class", "class_reason"])
       .agg(rows=("row_id", "size"), dollars=("amount", "sum")).reset_index())
cls["pct_rows"] = (100 * cls.rows / len(g)).round(2)
print("\n", cls.to_string(index=False))

## 11. Save outputs

| file | what | for |
|---|---|---|
| `pf_recipients_matched.parquet` | every grant row with class, EIN checks and match columns | task 4, ETL |
| `pf_recipients_for_fuzzy.parquet` | `ambiguous` + `unmatched` + `gov_name_review` rows | task 4 input |
| `reference_name_index.parquet` | normalized names of every known EIN (saved in step 4) | task 4 candidate pool |
| `bmf_clean.parquet`, `pub78_clean.parquet` | cleaned IRS references | everyone |
| `match_rates.csv`, `match_benchmarks.csv`, `recipient_classes.csv` | report tables | report / Curt follow-up |
| `spot_check_sample.csv` | up to 20 random rows per method | manual review |

In [ ]:
KEEP = ["row_id", "filer_ein", "taxyear", "line_number", "filer_state",
        "rcp_name1", "rcp_name2", "rcp_person", "name_from_person_field", "rcp_name_full", "rcp_name_norm",
        "name_maybe_truncated", "rcp_l1", "rcp_l2", "rcp_addr1", "rcp_city", "rcp_state", "rcp_zip",
        "rcp_city_n", "rcp_state_n", "rcp_zip5", "rcp_country", "rcp_status_txt", "amount", "purpose",
        "recipient_class", "class_reason", "gov_name_flag",
        "ein_raw", "ein_form", "ein_clean", "ein_status", "ein_in_bmf", "ein_in_pub78", "ein_in_990",
        "ein_in_990pf", "ein_in_revoked", "ein_name_score",
        "exact_ein", "exact_pass", "ein_digit_diff", "alt_ein", "n_candidates", "ambiguous_candidates",
        "match_method", "is_matched", "matched_ein", "matched_street", "matched_city", "matched_state",
        "matched_zip5", "matched_addr_source", "recipient_in_ma", "ma_flag_source"]
out = g[KEEP]
out.to_parquet(OUT_DIR / "pf_recipients_matched.parquet", index=False)
out[out.match_method.isin(["ambiguous", "unmatched", "gov_name_review"])].to_parquet(OUT_DIR / "pf_recipients_for_fuzzy.parquet", index=False)
match_rates.to_csv(OUT_DIR / "match_rates.csv", index=False)
bench.to_csv(OUT_DIR / "match_benchmarks.csv", index=False)
cls.to_csv(OUT_DIR / "recipient_classes.csv", index=False)

spot = out.sample(frac=1, random_state=539).groupby("match_method").head(SPOT_CHECK_PER_METHOD)
spot = spot.sort_values("match_method")
spot_cols = ["match_method", "recipient_class", "class_reason", "rcp_name_full", "rcp_name1", "rcp_name2", "rcp_person", "rcp_city",
             "rcp_state", "rcp_zip", "ein_raw", "ein_status", "ein_name_score", "matched_ein", "alt_ein", "matched_city",
             "matched_state", "ambiguous_candidates"]
spot[spot_cols].assign(reviewer_ok="", reviewer_note="").to_csv(OUT_DIR / "spot_check_sample.csv", index=False)

for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name:40s} {p.stat().st_size / 1e6:8.1f} MB")

## 12. Quick looks

Things worth eyeballing before the Curt follow-up: the most common unmatched names (often big national orgs
with chapter names, or schools/universities), and the most common EINs whose name disagrees.

In [ ]:
print("Top unmatched recipient names (rows, $):")
print(out[out.match_method == "unmatched"].groupby("rcp_name_norm")
      .agg(rows=("row_id", "size"), dollars=("amount", "sum")).sort_values("rows", ascending=False).head(25).to_string())

print("\nEINs whose reported name disagrees most often:")
dis = out[out.match_method == "ein_name_disagrees"]
print(dis.groupby(["ein_clean", "rcp_name_norm"]).size().sort_values(ascending=False).head(15).to_string())

print("\nEIN status by recipient class:")
print(pd.crosstab(out.recipient_class, out.ein_status.replace("", "(no EIN)")).to_string())